# iNaturalist Species Download

Downloads research-grade wildlife images from iNaturalist for all 7 target species.
Images are saved to Google Drive at `MyDrive/bc-dashcam/data/<species>/`.

**Source:** iNaturalist BC (place_id=7085), except cougar which uses global observations.

**Per-species caps:** the initial dataset capped every class at 500. The weak
classes (moose, elk, raccoon, cougar) get larger caps here to lift their mAP.
iNaturalist BC has far more available than 500 for every class except cougar
(deer ~30k, bear ~9.6k, raccoon ~4.4k, moose ~4.4k, coyote ~3.2k, elk ~2.1k,
cougar BC only 332 -> pulled globally). The download resumes where it left
off, so re-running with a higher cap only fetches the new images.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install pyinaturalist -q

In [ ]:
import requests
import os
from pyinaturalist import get_observations
import time
import json


def download_species(taxon_name, place_id, species_label, max_images=500, global_search=False):
    save_dir = f"/content/drive/MyDrive/bc-dashcam/data/{species_label}"
    os.makedirs(save_dir, exist_ok=True)

    existing = set(os.listdir(save_dir))
    downloaded = 0
    page = 1
    metadata = []

    while downloaded < max_images:
        print(f"{species_label} - Fetching page {page}...")
        try:
            kwargs = dict(
                taxon_name=taxon_name,
                quality_grade="research",
                photos=True,
                per_page=50,
                page=page
            )
            if not global_search:
                kwargs['place_id'] = place_id

            results = get_observations(**kwargs)
        except Exception as e:
            print(f"API call failed: {e}")
            break

        if not results['results']:
            print("No more results.")
            break

        for obs in results['results']:
            if downloaded >= max_images:
                break
            try:
                photo_url = obs['photos'][0]['url'].replace("square", "large")
                obs_id = obs['id']
                filename = f"{obs_id}.jpg"

                if filename in existing:
                    downloaded += 1
                    continue

                response = requests.get(photo_url, timeout=10)
                if response.status_code == 200:
                    with open(f"{save_dir}/{filename}", 'wb') as f:
                        f.write(response.content)
                    metadata.append({
                        'obs_id': obs_id,
                        'taxon': taxon_name,
                        'label': species_label,
                        'url': photo_url
                    })
                    downloaded += 1
                    if downloaded % 100 == 0:
                        print(f"  {species_label}: {downloaded}/{max_images}")
                time.sleep(0.3)
            except Exception as e:
                print(f"Error on {obs['id']}: {e}")
                continue

        page += 1
        time.sleep(1)

    meta_path = f"/content/drive/MyDrive/bc-dashcam/metadata/{species_label}_metadata.json"
    os.makedirs(os.path.dirname(meta_path), exist_ok=True)
    with open(meta_path, "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"Done: {species_label} - {downloaded} images saved to Drive.")
    return metadata

## Download all 7 species

One config table drives the whole pull. `cap` is the target image count per
class; `global_search=True` ignores the BC place filter (cougar only).

In [ ]:
# taxon, place_id, label, cap, global_search
SPECIES = [
    ("Alces alces",          7085, "moose",    1000, False),  # weak class, was 316
    ("Cervus canadensis",    7085, "elk",      1000, False),  # weak class, was 449
    ("Procyon lotor",        7085, "raccoon",  1000, False),  # weak class, was 397
    ("Puma concolor",        None, "cougar",   1000, True),   # global, BC only has 332
    ("Ursus americanus",     7085, "bear",      750, False),
    ("Canis latrans",        7085, "coyote",    750, False),
    ("Odocoileus hemionus",  7085, "deer",      750, False),
]

for taxon, place_id, label, cap, is_global in SPECIES:
    download_species(
        taxon_name=taxon,
        place_id=place_id,
        species_label=label,
        max_images=cap,
        global_search=is_global,
    )

## Verify download counts

In [ ]:
import os

for _, _, label, _, _ in SPECIES:
    path = f"/content/drive/MyDrive/bc-dashcam/data/{label}"
    if os.path.exists(path):
        print(f"{label}: {len(os.listdir(path))} images")
    else:
        print(f"{label}: folder missing")